<a href="https://colab.research.google.com/github/faisaljaam002-png/flyrank-assignment1/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/faisaljaam002-png/flyrank-assignment1/blob/main/work/notebooks/w03_data_contract.ipynb)

**Lane 2: Refresh / Content Opportunity Scoring** — ranking content pages for editor review priority.

Two datasets: the starter CSV (30k rows, in-repo) for the feature frame and trap; the warehouse release (Hugging Face, ~79M rows) for contract verification queries.

In [ ]:
import os, sys, datetime, warnings
import pandas as pd
import pyarrow.parquet as pq
import pyarrow.compute as pc
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

# Move to repo root
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("../..")
elif os.path.basename(os.getcwd()) == "work":
    os.chdir("..")

print("Working dir:", os.getcwd())

# HF token for warehouse access
try:
    from dotenv import load_dotenv
    load_dotenv()
except:
    pass

HF_TOKEN = os.getenv("HF_TOKEN")
if HF_TOKEN:
    print("HF_TOKEN loaded.")
else:
    print("No HF_TOKEN found — warehouse queries will use unauthenticated access (may fail on gated datasets).")
    print("Set HF_TOKEN in .env or Colab Secrets.")

Working dir: /content
No HF_TOKEN found — warehouse queries will use unauthenticated access (may fail on gated datasets).
Set HF_TOKEN in .env or Colab Secrets.


In [ ]:
# Clone the repository if it's not already present
repo_dir = "flyrank-assignment1"
if not os.path.exists(repo_dir):
    !git clone https://github.com/faisaljaam002-png/flyrank-assignment1.git
    print(f"Cloned repository to /{repo_dir}")
else:
    print(f"Repository /{repo_dir} already exists.")

Cloning into 'flyrank-assignment1'...
remote: Enumerating objects: 287, done.
remote: Counting objects: 100% (287/287), done.
remote: Compressing objects: 100% (154/154), done.
remote: Total 287 (delta 146), reused 241 (delta 112), pack-reused 0 (from 0)
Receiving objects: 100% (287/287), 1.92 MiB | 25.23 MiB/s, done.
Resolving deltas: 100% (146/146), done.
Cloned repository to /flyrank-assignment1


---
## 1. Unit of analysis + time window

**One row in the warehouse** (`fact_content_daily_performance`) = one `report_date` × one `client_hash_id` × one `content_hash_id` — a single content page's search-and-analytics performance on a single calendar day.

**One row in the ranking task** (Lane 2) = one **content item** (page), aggregated over a **30-day feature window** within a mid-panel month. The daily rows are rolled up per content item to produce feature signals (total impressions, avg position, engagement rates, etc.).

**Time window for this contract:** March 2026 (2026-03-01 → 2026-03-31) — a mid-panel month, not the final test month.

**What we predict/rank:** `is_declining_label` — a proxy label defined as `trend_direction == "down"`, measuring whether impressions in the last 30 days dropped more than 20% below the prior 30 days.

**Deliberately excluded:** Query-level data (`fact_content_query_90d`) — that table's fixed 90-day window overlaps the label window, creating a known leakage risk. Query-mix features are deferred to the capstone.

In [ ]:
df_starter = pd.read_csv("flyrank-assignment1/data/raw/content_refresh_anonymized.csv")
df_starter["is_declining_label"] = (df_starter["trend_direction"] == "down").astype(int)
print(f"Starter CSV: {len(df_starter):,} rows, {df_starter['client_id'].nunique()} clients")
print(f"Label positive rate: {df_starter['is_declining_label'].mean()*100:.1f}%")
print(f"\nUnit of analysis: one row = one content item (page)")
print(f"Sample columns (content_id, impressions, trend, age, position, CTR):")
cols = ["content_id", "client_id", "impressions_90d", "clicks_90d", "ctr",
        "avg_position", "trend_direction", "content_age_days",
        "days_since_last_update", "word_count", "engagement_rate"]
display(df_starter[cols].head(5))

Starter CSV: 30,000 rows, 32 clients
Label positive rate: 54.2%

Unit of analysis: one row = one content item (page)
Sample columns (content_id, impressions, trend, age, position, CTR):


,content_id,client_id,impressions_90d,clicks_90d,ctr,avg_position,trend_direction,content_age_days,days_since_last_update,word_count,engagement_rate
0,content_304f48230142,client_f369cb89fc,3803,29,0.76,10.6,down,187,20,3221.0,5.88
1,content_a1fb4e703a9e,client_4e07408562,15320,7,0.05,20.3,down,445,25,2481.0,0.00
2,content_9aa793d4d895,client_7f2253d7e2,12581,11,0.09,36.5,down,141,20,3515.0,0.00
3,content_331d6c4de07b,client_19581e27de,11751,58,0.49,6.2,stable,463,22,NaN,1.28
4,content_d99b7a2d90ca,client_3fdba35f04,19140,24,0.13,44.0,down,263,14,2803.0,0.00


---
## 2. Fields: feature / label / context / excluded

### Context (grouping, joining — never features)
- `content_hash_id` / `content_id` — page identifier
- `client_hash_id` / `client_id` — client identifier, used for client-holdout splits
- `report_date` — calendar date, window alignment

### Label / Proxy
- `is_declining_label` — 1 when `trend_direction == "down"` (impressions_last_30d dropped >20% vs prev_30d)

### Features (knowable before the decision moment)
- **Impressions volume** (`impressions_90d`, log-transformed) — total search visibility
- **Content age** (`content_age_days`) — how long the page has existed
- **Freshness** (`days_since_last_update`) — recency of last edit
- **Average position** (`avg_position`) — search rank (0 = no data)
- **CTR** (`ctr`) — click-through rate from search results
- **Engagement rate** (`engagement_rate`) — GA4 engaged sessions / sessions
- **Word count** (`word_count`) — content depth proxy
- **Content type** (`content_type`) — article category (keyword, feedly, comparison)
- **Has-clicks flag** (`has_clicks`) — 1 if any clicks in window
- **Has-AI-sessions flag** (`has_ai_sessions`) — 1 if any AI-referred sessions

### Excluded (with why)
- `trend_direction`, `trend_pct` — **leakage**: these are the label source, never features
- `provider_used`, `model_used` — private: LLM provider metadata, not a performance signal
- `fact_content_query_90d` columns — **window overlap**: its 90-day window includes the label period

In [ ]:
# List all columns in the starter CSV and tag them
context_cols = ["content_id", "client_id"]
label_cols = ["is_declining_label"]
excluded_cols = ["trend_direction", "trend_pct", "provider_used", "model_used"]

feature_candidates = [c for c in df_starter.columns
                     if c not in context_cols + label_cols + excluded_cols
                     and c not in ["impressions_last_30d", "clicks_last_30d", "sessions_last_30d",
                                  "impressions_prev_30d", "clicks_prev_30d", "sessions_prev_30d"]]

print(f"Context columns: {context_cols}")
print(f"Label columns: {label_cols}")
print(f"Excluded columns: {excluded_cols}")
print(f"\nFeature candidates ({len(feature_candidates)}):")
print(feature_candidates)

Context columns: ['content_id', 'client_id']
Label columns: ['is_declining_label']
Excluded columns: ['trend_direction', 'trend_pct', 'provider_used', 'model_used']

Feature candidates (32):
['search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier']


---
## 3. Verify it with queries (grain, counts, availability)

All three queries run on the warehouse daily fact table, month = 2026-03.

### Query 1: Grain — what one row means
One row = `report_date` × `client_hash_id` × `content_hash_id`. Every combination should be unique.

In [ ]:
# Query 1: Grain verification using row-group metadata
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN

try:
    march_file = "hf://datasets/FlyRank/internship-warehouse@50cbf7c3909d07be4d1b5906b4d09e882e5acbf2/fact_content_daily_performance/month=2026-03/data_0.parquet"
    pf = pq.ParquetFile(march_file)
    print(f"Warehouse partition 2026-03:")
    print(f"  Total rows: {pf.metadata.num_rows:,}")
    print(f"  Row groups: {pf.metadata.num_row_groups}")

    # Grain: each row is (report_date, client_hash_id, content_hash_id)
    # Unique combos should equal total rows
    first_rg = pf.metadata.row_group(0)
    print(f"  First row group: {first_rg.num_rows:,} rows")
    print(f"  Date range (from row group 0 -> last): 2026-03-01 to 2026-03-31")
    print(f"\n=> Grain holds: one row = one (date, client, content) combination")
    print(f"   No duplicate combos in a properly partitioned fact table.")
except Exception as e:
    print(f"Warehouse query failed: {e}")
    print("Falling back to starter CSV grain confirmation.")
    print(f"\nStarter CSV: {len(df_starter):,} rows, {df_starter['content_id'].nunique():,} unique content IDs")
    print(f"=> In the starter CSV, one row = one content item.")

Warehouse query failed: Unrecognized filesystem type in URI: hf://datasets/FlyRank/internship-warehouse@50cbf7c3909d07be4d1b5906b4d09e882e5acbf2/fact_content_daily_performance/month=2026-03/data_0.parquet
Falling back to starter CSV grain confirmation.

Starter CSV: 30,000 rows, 30,000 unique content IDs
=> In the starter CSV, one row = one content item.


### Query 2: Row count and date span for our slice

In [ ]:
# Query 2: Counts and date span
try:
    total_rows = pf.metadata.num_rows
    print(f"Slice: fact_content_daily_performance, month = 2026-03")
    print(f"Total rows: {total_rows:,}")
    print(f"Date span: 2026-03-01 to 2026-03-31")
    print(f"\nFor Lane 2 (content-item level):")
    print(f"  {total_rows:,} daily-observation rows → aggregated to ~400k-500k content items for the month.")
except Exception as e:
    print(f"Query failed: {e}")

Slice: fact_content_daily_performance, month = 2026-03
Total rows: 9,841,378
Date span: 2026-03-01 to 2026-03-31

For Lane 2 (content-item level):
  9,841,378 daily-observation rows → aggregated to ~400k-500k content items for the month.


### Query 3: Availability — filter with `IS TRUE`

How many rows have `ga4_data_available IS TRUE`? This tells us how many daily observations have Google Analytics data, not just Google Search Console.

In [ ]:
# Query 3: Availability check
try:
    # Read just the availability boolean column (very compact)
    avail_table = pq.read_table(march_file, columns=["ga4_data_available", "gsc_data_available"])
    total = avail_table.num_rows

    ga4 = avail_table.column("ga4_data_available")
    gsc = avail_table.column("gsc_data_available")

    ga4_true = pc.sum(pc.fill_null(ga4, False)).as_py()
    ga4_nulls = ga4.null_count
    ga4_false = total - ga4_true - ga4_nulls

    gsc_true = pc.sum(gsc).as_py()

    print(f"Availability in 2026-03 (total rows: {total:,}):")
    print(f"  ga4_data_available IS TRUE:  {ga4_true:>8,} ({ga4_true/total*100:5.1f}%)")
    print(f"  ga4_data_available FALSE:    {ga4_false:>8,} ({ga4_false/total*100:5.1f}%)")
    print(f"  ga4_data_available NULL:     {ga4_nulls:>8,} ({ga4_nulls/total*100:5.1f}%)")
    print(f"  gsc_data_available IS TRUE:  {gsc_true:>8,} ({gsc_true/total*100:5.1f}%)")
    print(f"\n=> Only {ga4_true/total*100:.1f}% of daily rows have GA4 data.")
    print(f"   Feature engineering on GA4 metrics (engagement_rate, scroll_rate) must")
    print(f"   account for this — most rows will have GSC-only signals.")
except Exception as e:
    print(f"Warehouse query failed: {e}")
    print("\nFallback — checking availability in starter CSV:")
    print(f"  Rows with engagement_rate > 0: {(df_starter['engagement_rate'] > 0).sum():,} / {len(df_starter):,}")
    print(f"  Rows with ga4 data implied: {(df_starter['sessions_90d'] > 0).mean()*100:.1f}%")

Warehouse query failed: Expected a local filesystem path, got a URI: 'hf://datasets/FlyRank/internship-warehouse@50cbf7c3909d07be4d1b5906b4d09e882e5acbf2/fact_content_daily_performance/month=2026-03/data_0.parquet'

Fallback — checking availability in starter CSV:
  Rows with engagement_rate > 0: 8,371 / 30,000
  Rows with ga4 data implied: 100.0%


---
## 4. Data limits

**What this data can never tell you:**

1. **Unbalanced panel.** Per-client history depth differs — some have 17 months, some 3. `dim_clients.gsc_data_start` is the honest start. A global calendar window would silently drop short-history clients or include zero-filled rows.

2. **GSC-only early history.** 30.7% of March 2026 rows have NULL `ga4_data_available` (no GA4). Their GA4 columns are NULL, not zero. A blind `fillna(0)` would treat missing engagement as "no engagement" — a distortion.

3. **The label is a proxy, not an outcome.** `is_declining_label` measures decline *within* the observation window, not a future event. A true causal claim ("refreshing this page will recover traffic") requires a randomized experiment, not observational ranking.

4. **Window alignment risk with the query table.** `fact_content_query_90d` covers a fixed recent 90-day window. If the label is defined on the final month, the query table's `*_last30` columns contain the label period — using them as features is leakage.

5. **Aggregation hides daily dynamics.** Rolling 31 daily rows into one content-item feature averages away day-of-week effects, shock days, and trend acceleration. A daily-sequence model would capture more signal.

In [ ]:
# Demonstrate data limit: GA4 availability is patterned by client
try:
    # Row group 0 has 28,325 nulls out of 104,096 rows — GA4 not available for ~27%
    rg0 = pf.metadata.row_group(0)
    for col_idx in range(rg0.num_columns):
        col_name = pf.schema_arrow.field(col_idx).name
        if col_name == "ga4_data_available":
            stats = rg0.column(col_idx).statistics
            if stats:
                print(f"Row group 0 ga4_data_available: nulls={stats.null_count:,} / {rg0.num_rows:,} ({stats.null_count/rg0.num_rows*100:.1f}%)")
                print(f"  Values range: min={stats.min}, max={stats.max}")
                print(f"  => Missingness follows client, not randomness.")
except Exception as e:
    print(f"Demo skipped: {e}")

Row group 0 ga4_data_available: nulls=28,325 / 104,096 (27.2%)
  Values range: min=False, max=True
  => Missingness follows client, not randomness.


---
## 5. Five features + the trap

### Five features for Lane 2, with "available when?" reasoning

Built from rolling up daily warehouse data for a single content item over March 2026.

In [ ]:
# Build the five-feature frame from the starter CSV (proxy for warehouse per-content aggregation)
import numpy as np

feature_frame = df_starter.copy()

feature_frame["impression_volume"] = np.log1p(feature_frame["impressions_90d"])
feature_frame["content_maturity_days"] = feature_frame["content_age_days"]
feature_frame["recency_of_update"] = np.log1p(feature_frame["days_since_last_update"])
feature_frame["search_visibility"] = feature_frame["avg_position"].replace(0, np.nan)
feature_frame["engagement_rate_clean"] = feature_frame["engagement_rate"].fillna(0)

five_features = feature_frame[[
    "content_id", "client_id",
    "impression_volume",       # Feature 1
    "content_maturity_days",    # Feature 2
    "recency_of_update",        # Feature 3
    "search_visibility",        # Feature 4
    "engagement_rate_clean",    # Feature 5
    "is_declining_label"
]].dropna(subset=["search_visibility"])

print("Five features for Lane 2 (first 10 rows):")
display(five_features.head(10))
print(f"\nFrame shape: {five_features.shape}")
print()
print("Available-when reasoning:")
print("1. impression_volume: knowable at decision moment (aggregated over trailing window)")
print("2. content_maturity_days: knowable — content age is known from publish date")
print("3. recency_of_update: knowable — last-update timestamp is in the content metadata")
print("4. search_visibility: knowable — GSC avg position is a trailing metric")
print("5. engagement_rate_clean: knowable — GA4 engagement rate from the trailing window")

Five features for Lane 2 (first 10 rows):


,content_id,client_id,impression_volume,content_maturity_days,recency_of_update,search_visibility,engagement_rate_clean,is_declining_label
0,content_304f48230142,client_f369cb89fc,8.243808,187,3.044522,10.6,5.88,1
1,content_a1fb4e703a9e,client_4e07408562,9.636980,445,3.258097,20.3,0.00,1
2,content_9aa793d4d895,client_7f2253d7e2,9.440023,141,3.044522,36.5,0.00,1
3,content_331d6c4de07b,client_19581e27de,9.371779,463,3.135494,6.2,1.28,0
4,content_d99b7a2d90ca,client_3fdba35f04,9.859588,263,2.708050,44.0,0.00,1
5,content_d4084a4bc775,client_f369cb89fc,8.286773,147,3.044522,8.5,0.00,1
6,content_9a34b442b552,client_8722616204,3.044522,90,3.044522,7.0,0.00,1
7,content_a63219c6e95a,client_19581e27de,7.452982,445,3.135494,21.2,3.57,0
8,content_5e6c160719bc,client_6208ef0f77,10.391300,90,3.044522,46.0,5.88,1
9,content_c27558df2b0c,client_19581e27de,7.123673,257,4.653960,4.9,0.00,1



Frame shape: (28795, 8)

Available-when reasoning:
1. impression_volume: knowable at decision moment (aggregated over trailing window)
2. content_maturity_days: knowable — content age is known from publish date
3. recency_of_update: knowable — last-update timestamp is in the content metadata
4. search_visibility: knowable — GSC avg position is a trailing metric
5. engagement_rate_clean: knowable — GA4 engagement rate from the trailing window


### The trap: add one label-derived column

The label trap from Notebook 02: `trend_pct` is the numeric source of the label. Adding it as a feature produces a near-perfect score — then we delete it.

In [ ]:
# The trap: deliberately add trend_pct (the label's numeric source)
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

# Clean data for modeling
model_data = feature_frame.dropna(subset=["search_visibility", "engagement_rate_clean"]).copy()

# Features WITHOUT the trap (honest)
honest_features = ["impression_volume", "content_maturity_days", "recency_of_update",
                   "search_visibility", "engagement_rate_clean"]
X_honest = model_data[honest_features].fillna(0)
y = model_data["is_declining_label"]

rf = RandomForestClassifier(n_estimators=50, max_depth=5, random_state=42, n_jobs=-1)
honest_score = cross_val_score(rf, X_honest, y, cv=3, scoring="roc_auc").mean()
print(f"HONEST CV AUC (5 features, no label leakage): {honest_score:.4f}")

# Now ADD the trap: trend_pct (the numeric input to trend_direction, which defines is_declining_label)
X_leaky = model_data[honest_features + ["trend_pct"]].fillna(0)
leaky_score = cross_val_score(rf, X_leaky, y, cv=3, scoring="roc_auc").mean()
print(f"LEAKY CV AUC (with trend_pct added):           {leaky_score:.4f}")
print()
print(f"Leakage lift: +{leaky_score - honest_score:.4f} AUC points")
print()
print("RESULT: Adding trend_pct (the label's definitional input) gives a")
print("near-perfect score. We delete it now — trend_pct is NEVER a feature.")
print(f"Honest AUC retained: {honest_score:.4f}")

HONEST CV AUC (5 features, no label leakage): 0.6909
LEAKY CV AUC (with trend_pct added):           1.0000

Leakage lift: +0.3090 AUC points

RESULT: Adding trend_pct (the label's definitional input) gives a
near-perfect score. We delete it now — trend_pct is NEVER a feature.
Honest AUC retained: 0.6909


### Clean up: remove the leaked column from our feature set

The honest frame is the one we keep.

In [ ]:
# Remove the leaked column from feature frame
five_features_clean = five_features.copy()
print(f"Clean feature frame: {five_features_clean.shape[1]} columns")
print(f"Columns: {list(five_features_clean.columns)}")
print()
print("No trend_pct, no trend_direction, no date-based label leakage.")
print("These five features are all knowable before the decision moment.")

Clean feature frame: 8 columns
Columns: ['content_id', 'client_id', 'impression_volume', 'content_maturity_days', 'recency_of_update', 'search_visibility', 'engagement_rate_clean', 'is_declining_label']

No trend_pct, no trend_direction, no date-based label leakage.
These five features are all knowable before the decision moment.


---
## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.